# Importing Libraries

In [1]:
import requests
import json
import pandas as pd
import openpyxl
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
import csv
import re

In [ ]:

master_url = "https://eraktkosh.mohfw.gov.in/eraktkoshPortal/eraktkosh/master/all"
stock_url = "https://eraktkosh.mohfw.gov.in/eraktkoshPortal/eraktkosh/blood-availability" 

{'statesWithDistricts': [{'stateCode': '35', 'stateName': 'Andaman and Nicobar Islands', 'districts': [{'districtCode': '638', 'districtName': 'Nicobars'}, {'districtCode': '639', 'districtName': 'North  & Middle Andaman'}, {'districtCode': '640', 'districtName': 'South Andaman'}]}, {'stateCode': '28', 'stateName': 'Andhra Pradesh', 'districts': [{'districtCode': '679', 'districtName': 'Alluri Sitharama Raju'}, {'districtCode': '680', 'districtName': 'Anakapalli'}, {'districtCode': '553', 'districtName': 'Ananthapuramu'}, {'districtCode': '753', 'districtName': 'Annamayya'}, {'districtCode': '750', 'districtName': 'Bapatla'}, {'districtCode': '554', 'districtName': 'Chittoor'}, {'districtCode': '682', 'districtName': 'Dr. B. R. Ambedkar Konaseema'}, {'districtCode': '545', 'districtName': 'East Godavari'}, {'districtCode': '748', 'districtName': 'Eluru'}, {'districtCode': '548', 'districtName': 'Guntur'}, {'districtCode': '681', 'districtName': 'Kakinada'}, {'districtCode': '547', 'dis

# Initialization of Main Function

In [26]:
def fetch_master_all():
    headers = {
    'Content-Type' : 'application/json'
    }
    response = requests.post(master_url,json={"hospitalCode": 100}, headers=headers)
    response.raise_for_status()
    payload = response.json()
    state_dict = {}
    district_dict = {}
    for state in payload.get("statesWithDistricts", []):
        state_code = state["stateCode"]
        state_dict[state["stateName"]] = state_code
        district_dict[state_code] = {
            d["districtName"]: d["districtCode"] for d in state.get("districts", [])
        }
    blood_dict = {g["bloodGroupName"]: g["bloodGroupCode"] for g in payload.get("bloodGroups", [])}
    component_dict = {c["componentName"]: c["componentCode"] for c in payload.get("componentList", [])}
    return state_dict, district_dict, blood_dict, component_dict

In [27]:
def save_master_data(path="master_data.json"):
    """Cache master data to disk so you don't refetch it on every collection run."""
    state_dict, district_dict, blood_dict, component_dict = fetch_master_all()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "state_dict": state_dict,
                "district_dict": district_dict,
                "blood_dict": blood_dict,
                "component_dict": component_dict,
            },
            f, ensure_ascii=False, indent=2,
        )
    return state_dict, district_dict, blood_dict, component_dict

In [28]:
def load_master_data(path="master_data.json"):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["state_dict"], data["district_dict"], data["blood_dict"], data["component_dict"]

In [ ]:
def fetch_blood_data(state_code, district_code, blood_group_code, component_code, component_name):
    params = {
        "stateCode": state_code,
        "districtId": district_code,
        "componentId": component_code,
        "bloodGroupId": blood_group_code,
    }
    response = requests.get(stock_url, params=params)
    response.raise_for_status()
    entries = response.json()
    fetched_at = datetime.now().isoformat(timespec="seconds")
    cleaned = []
    for entry in entries:
        comp_info = entry.get("components", {}).get(component_name, {})
        cleaned.append({
            "fetched_at": fetched_at,
            "state_code": state_code,
            "district_code": district_code,
            "blood_group": blood_group_code,
            "blood_component": component_code,
            "blood_bank": entry.get("hospitalname"),
            "hospital_code": entry.get("hospitalCode"),
            "address": entry.get("hospitaladd"),
            "contact": entry.get("hospitalcontact"),
            "category": entry.get("hospitalType"),
            "available": comp_info.get("available_WithQty", ""),
            "not_available": comp_info.get("not_available_WithQty", ""),
            "last_updated": entry.get("entrydate"),
            "bank_type": entry.get("type"),
        })
    return cleaned